# Task 12: High-Dimensional Latent Space Anomaly Detection via Convolutional Autoencoders

**Objective:** Identify structural anomalies and defects in fabric/manufacturing images by training a deep convolutional autoencoder on fault-free datasets and evaluating SSIM reconstruction error maps.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# 1. SSIM Loss function from scratch (Simplified 2D implementation using local kernels)
def calculate_ssim(x, y, window_size=11, eps=1e-8):
    # x, y shape: (B, 1, H, W)
    C1 = (0.01 * 2.0) ** 2
    C2 = (0.03 * 2.0) ** 2
    
    # Local mean kernel
    kernel = torch.ones(1, 1, window_size, window_size, device=x.device) / (window_size ** 2)
    
    mu1 = F.conv2d(x, kernel, padding=window_size//2)
    mu2 = F.conv2d(y, kernel, padding=window_size//2)
    
    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2
    
    sigma1_sq = F.conv2d(x * x, kernel, padding=window_size//2) - mu1_sq
    sigma2_sq = F.conv2d(y * y, kernel, padding=window_size//2) - mu2_sq
    sigma12 = F.conv2d(x * y, kernel, padding=window_size//2) - mu1_mu2
    
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return torch.clamp(ssim_map, min=-1.0, max=1.0)

# 2. Convolutional Autoencoder with deep bottleneck
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: compress image down to bottleneck size
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), # 128x128 -> 64x64
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), # 64x64 -> 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # 32x32 -> 16x16
            nn.ReLU(),
            nn.Conv2d(64, 8, 3, stride=1, padding=1)  # Deep Bottleneck (8 feature channels)
        )
        # Decoder: reconstruct back
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 64, 3, stride=2, padding=1, output_padding=1), # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), # 32x32 -> 64x64
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), # 64x64 -> 128x128
            nn.ReLU(),
            nn.Conv2d(16, 1, 3, padding=1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

In [ ]:
# Setup mock training and anomaly evaluation pipeline
autoencoder = ConvAutoencoder()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

# Generate mock defect-free training images (normal horizontal grid lines)
normal_images = []
for _ in range(20):
    grid = np.zeros((1, 128, 128), dtype=np.float32)
    grid[0, ::8, :] = 1.0 # Draw clean grid
    normal_images.append(torch.tensor(grid))
normal_loader = torch.stack(normal_images)

# Simple training loop
for epoch in range(10):
    reconstructed = autoencoder(normal_loader)
    # Composite Loss: MSE + SSIM
    mse = F.mse_loss(reconstructed, normal_loader)
    ssim = 1.0 - calculate_ssim(reconstructed, normal_loader).mean()
    loss = mse + 0.5 * ssim
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
# Evaluate an anomaly (introducing random blobs/scratches on normal grids)
normal_test = normal_loader[0:1]
anomaly_test = normal_loader[0:1].clone()
anomaly_test[0, 0, 40:60, 40:60] = 0.5 # Add defect

with torch.no_grad():
    rec_normal = autoencoder(normal_test)
    rec_anomaly = autoencoder(anomaly_test)
    
    # Compute reconstruction error map
    err_normal = (normal_test - rec_normal).pow(2)
    err_anomaly = (anomaly_test - rec_anomaly).pow(2)

print("Defect-free Reconstruction Loss (MSE):", err_normal.mean().item())
print("Anomalous Image Reconstruction Loss (MSE): ", err_anomaly.mean().item())
assert err_anomaly.mean().item() > err_normal.mean().item(), "Anomaly detection failed!"
print("Success: Anomaly autoencoder correctly flags structural defects!")